In [1]:
cd /gpfs/group/smr48/default/io

/gpfs/group/smr48/default/io


In [13]:
import pandas as pd
import numpy as np

In [18]:
length = 0
masterTweets = pd.read_csv('_masterTweetList_2021-03.csv', chunksize=20000000, dtype={'m_tweetid':str})
for chunk in masterTweets: 
    c_len = len(chunk)
    length += c_len
    print(length)
    print(chunk.info())

20000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000000 entries, 0 to 19999999
Data columns (total 3 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   m_tweetid  object
 1   m_userid   object
 2   m_country  object
dtypes: object(3)
memory usage: 457.8+ MB
None
40000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000000 entries, 20000000 to 39999999
Data columns (total 3 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   m_tweetid  object
 1   m_userid   object
 2   m_country  object
dtypes: object(3)
memory usage: 457.8+ MB
None
60000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000000 entries, 40000000 to 59999999
Data columns (total 3 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   m_tweetid  object
 1   m_userid   object
 2   m_country  object
dtypes: object(3)
memory usage: 457.8+ MB
None
80000000
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000000 entries, 60000000 to 79999999
Data columns (total 3 columns):
 

In [54]:
chunk[4800000:4900000]

,m_tweetid,m_userid,m_country
204800000,1118555721954729984,pBI79nNZ8ZvmEcfND9boXzAO81sW0hju6LIuZe8YPuo=,uae_egypt
204800001,1110375822622220288,981925161271250945,uae_egypt
204800002,1153228187469582337,981925161271250945,uae_egypt
204800003,1067107628948619266,981925161271250945,uae_egypt
204800004,1118476219627057153,981925161271250945,uae_egypt
...,...,...,...
204899995,5.492105986683863e+17,2547578185,venezuela
204899996,5.6633375389178266e+17,2547578185,venezuela
204899997,5.630004549727191e+17,2547578185,venezuela
204899998,5.630495530843095e+17,2547578185,venezuela


In [17]:
masterTweets2 = pd.read_csv('_masterTweetList_2021-03.csv', nrows = 10000, dtype={'m_tweetid':str})
masterTweets2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   m_tweetid  10000 non-null  object
 1   m_userid   10000 non-null  object
 2   m_country  10000 non-null  object
dtypes: object(3)
memory usage: 234.5+ KB


In [ ]:
folder = 'armenia'
number = 1

import pandas as pd
import os

df = pd.read_pickle('_Parsed/{}_parsed_{}.pkl'.format(folder, number))

df.tweetid = df.tweetid.apply(str)
df.userid = df.userid.apply(str)
df.reply_user_id = df.reply_user_id.apply(str)
df.retweet_user_id = df.retweet_user_id.apply(str)

# Load all Tweets completed by IO actors
# only load first 1000000 for testing
masterTweets = pd.read_csv('_masterTweetList_2021-03.csv', nrows=50000000)
masterTweets["m_userid"]= masterTweets["m_userid"].astype(str) 
masterTweets["m_tweetid"]= masterTweets["m_tweetid"].astype(str) 

"""CONFIRM CSV DATE"""
masterUIDs = pd.read_csv('_masterUIDs_2021-03.csv')
masterUIDs = masterUIDs[masterUIDs['id']!='userid']
masterUIDs.id = masterUIDs.id.apply(str)


####### RETWEET Actors #######
print('working retweets')
dfRT = df[df['is_retweet'] != 'false'] # removes all Tweets that are not RTs
dfRT = dfRT[dfRT['is_retweet'] != 'FALSE']
dfRT2 = df[df['retweet_user_id'] != ''] # removes all Tweets that are not retweets
dfRT3 = df[df['retweet_tweet_id'] != ''] # removes all Tweets that are not retweets
dfRT = pd.concat([dfRT, dfRT2, dfRT3])
dfRT = dfRT.drop_duplicates(subset='tweetid', keep='first')
dfRT['retweet_tweet_id'] = dfRT.retweet_tweet_id.apply(str)
dfRT = pd.merge(dfRT,masterTweets,left_on='retweet_tweet_id',right_on='m_tweetid',how= 'left')
print('dfRT len: {}'.format(len(dfRT)))
# Merge will help identify if any parent RT is made by an IO actor in collection..based off TweetID. 

dfRT_edges1 = dfRT.copy()
dfRT_edges1 = dfRT_edges1[~dfRT_edges1['m_userid'].isnull()]
dfRT_edges1 = dfRT_edges1[['tweetid', 'userid', 'm_userid', 'tweet_time']]
dfRT_edges1 = dfRT_edges1.drop_duplicates()


dfRT['retweet_user_id'] = dfRT.retweet_user_id.apply(str)
dfRT = pd.merge(dfRT,masterUIDs,left_on='retweet_user_id',right_on='id',how= 'left')
print('dfRT len: {}'.format(len(dfRT)))

# Merge will help identify if any parent RT is made by an IO actor in collection..based off UserID. 

dfRT_edges2 = dfRT.copy()
dfRT_edges2 = dfRT_edges2[~dfRT_edges2['id'].isnull()]
dfRT_edges2 = dfRT_edges2[['tweetid', 'userid', 'id', 'tweet_time']]
dfRT_edges2 = dfRT_edges2.drop_duplicates()

dfRT_edges1.columns = ['tweetid', 'userid', 'parentID', 'tweet_time']
dfRT_edges2.columns = ['tweetid', 'userid', 'parentID', 'tweet_time']
RT_edges = pd.concat([dfRT_edges1, dfRT_edges2])
RT_edges = RT_edges.drop_duplicates()
RT_edges['edge_type'] = 'retweet'
print('RTedges: {}'.format(len(RT_edges)))


####### REPLY Actors #######
print('working replies')
dfRply = df[df['reply_user_id'] != ''] # removes all Tweets that are not replies
dfRply2 = df[df['reply_tweet_id'] != '']
dfRply = pd.concat([dfRply, dfRply2])
dfRply = dfRply.drop_duplicates(subset='tweetid', keep='first')

dfRply['reply_tweet_id'] = dfRply.reply_tweet_id.apply(str)
dfRply['reply_user_id'] = dfRply.reply_user_id.apply(str)
dfRply = pd.merge(dfRply, masterTweets, left_on='reply_tweet_id', right_on='m_tweetid', how='left')
print('dfRP len: {}'.format(len(dfRply)))

dfRply = pd.merge(dfRply, masterUIDs, left_on='reply_user_id', right_on='id', how='left')
print('dfRP len: {}'.format(len(dfRply)))

RP_edges1 = dfRply[~dfRply['id'].isnull()]
RP_edges1 = RP_edges1[['tweetid', 'userid', 'id', 'tweet_time']]
RP_edges1.columns = ['tweetid', 'userid', 'parentID', 'tweet_time']
RP_edges2 = dfRply[~dfRply['m_userid'].isnull()]
print('dfRP len: {}'.format(len(RP_edges2)))
RP_edges2 = RP_edges2[['tweetid', 'userid', 'm_userid', 'tweet_time']]
RP_edges2.columns = ['tweetid', 'userid', 'parentID', 'tweet_time']
RP_edges = pd.concat([RP_edges1, RP_edges2])
RP_edges = RP_edges.drop_duplicates()
RP_edges['edge_type'] = 'reply'
print('RPedges: {}'.format(len(RP_edges)))




####### QUOTE Actors #######
print('working quotes')
dfQT = df[df['quote_tweet_id'] != ''] # removes all Tweets that are not quotes
# print('Len of dfQT: {}\n'.format(len(dfQT)))
dfQT['quote_tweet_id'] = dfQT.quote_tweet_id.apply(str)

dfQT = pd.merge(dfQT,masterTweets,left_on='quote_tweet_id',right_on='m_tweetid',how= 'inner')
dfQT = dfQT.drop_duplicates()
dfQT = dfQT[~dfQT['m_userid'].isnull()]
QT_edges = dfQT[['tweetid', 'userid', 'm_userid', 'tweet_time']]
QT_edges.columns = ['tweetid', 'userid', 'parentID', 'tweet_time']
QT_edges['edge_type'] = 'quote'
print('QTedges: {}'.format(len(QT_edges)))









####### MENTIONED Actors #######
print('working Mentions')
df['mentions'] = df['mentions'].str.replace(']', '')
df['mentions'] = df['mentions'].str.replace('[', '')
df['mentions'] = df['mentions'].str.replace(' ', '')
df['mentions'] = df['mentions'].str.replace('\'', '')
df2 = df.copy()
df2 = df2.join(df2.pop('mentions').str.split(',',expand=True))
df2.set_index(['tweetid','userid', 'accountDate', 'account_lang', 'tweet_lang',
              'text', 'tweet_time', 'tweetClient', 'reply_user_id', 'reply_tweet_id',
              'quote_tweet_id', 'is_retweet', 'retweet_user_id', 'retweet_tweet_id',
              'quote_count', 'reply_count', 'like_count', 'retweet_count', 'hashtags',
              'urls']).stack().reset_index(level=[0,1],name='mention')



df_melt = df2.melt(id_vars =['tweetid','userid','accountDate',
                            'account_lang','tweet_lang','text',
                            'tweet_time','tweetClient','reply_user_id',
                            'reply_tweet_id','quote_tweet_id','is_retweet',
                            'retweet_user_id','retweet_tweet_id','quote_count',
                             'reply_count', 'like_count', 'retweet_count', 'hashtags','urls']).dropna()

df_melt = df_melt.rename(columns={'value': 'id'})
df_melt.id = df_melt.id.apply(str)


dfMT = pd.merge(df_melt, masterUIDs, on=['id'], how='left')

MT_edges = dfMT[~dfMT['country'].isnull()]
MT_edges = MT_edges.drop_duplicates()
MT_edges = MT_edges[['tweetid', 'userid', 'id', 'tweet_time']]
MT_edges.columns = ['tweetid', 'userid', 'parentID', 'tweet_time']
MT_edges['edge_type'] = 'mention'
print('MTedges: {}'.format(len(MT_edges)))

print('Merging all edges')
IO_edges = pd.concat([RT_edges, RP_edges, QT_edges, MT_edges])
print('IOedges: {}'.format(len(IO_edges)))
IO_edges = IO_edges.drop_duplicates()
print('IOedges_post_dedup: {}'.format(len(IO_edges)))

# IO_edges.to_csv('_Edges/_{}_IO_edges_{}.csv'.format(folder, number), index = False) 

print('\n\nfinished')